In [0]:
df_gold_cost_by_service = spark.sql("""
    SELECT
        RECORD_TYPE,
        CHARGE_CODE,
        CATEGORY,
        
        SUM(QUANTITY)       as TOTAL_QUANTITY,
        SUM(VENDOR_COST)    as TOTAL_VENDOR_COST,
        SUM(TOTAL_CHARGES)  as TOTAL_CHARGES
    FROM (
        SELECT RECORD_TYPE, CHARGE_CODE, CATEGORY, QUANTITY, VENDOR_COST, TOTAL_CHARGES
        FROM finops.silver.focus

        UNION ALL

        SELECT RECORD_TYPE, CHARGE_CODE, CATEGORY, QUANTITY, VENDOR_COST, TOTAL_CHARGES
        FROM finops.silver.non_focus
    )
    GROUP BY RECORD_TYPE, CHARGE_CODE, CATEGORY
    ORDER BY RECORD_TYPE, CHARGE_CODE, CATEGORY
""")

print(f"✅ Gold aggregation done — {df_gold_cost_by_service.count()} rows")
df_gold_cost_by_service.show(10)

In [0]:
df_gold_cost_by_service.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://gold@adbstoragev10.dfs.core.windows.net/cost_by_service")